## Library imports and set params

In [ ]:
# Standard library imports
import os
from glob import glob
from datetime import datetime
from pathlib import Path

# Data science and scientific libraries
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.signal import savgol_filter
from shapely.geometry import LineString, Point, MultiPoint
from shapely.ops import substring, voronoi_diagram
from shapely.affinity import rotate, translate
import rivabar as rb


# Plotting libraries
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.transforms as transforms
from matplotlib.lines import Line2D

# Use seaborn base theme
import seaborn as sns
sns.set_theme(style="white") 

# Set fonts for manuscript figures (AGU / journal standards)
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Helvetica']
mpl.rcParams['pdf.fonttype'] = 3
mpl.rcParams['ps.fonttype'] = 3
mpl.set_loglevel("critical")

### Define directory structure relative to this notebook

In [ ]:
# Resolve the current directory robustly (works for both notebooks and scripts)
notebook_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# Define project root (one level up from 'notebooks')
root_dir = notebook_dir.parent

# Define input/output directories relative to the root
data_dir = root_dir / "data"
fig_dir = root_dir / "figures"
table_dir = root_dir / "tables"

# Create the figures directory if it does not already exist
data_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

# Print paths to verify successful setup
print(f"Project Root:  {root_dir}")
print(f"Data Dir:      {data_dir}")
print(f"Figures Dir:   {fig_dir}")
print(f"Tables Dir:    {table_dir}")

## Import and view discharge data

In [ ]:
romayor = pd.read_csv(data_dir / 'discharge' / 'trinity_discharge_romayor.csv', sep=',', skiprows=29)

romayor = romayor.rename(columns={
    "133980_00060_00003": "Q_cfps_mean",
    "133980_00060_00003_cd": "Q_cfps_mean_status",
    "133985_00065_00003": "GH_ft_mean",
    "133985_00065_00003_cd": "GH_ft_mean_status",
    "173616_00060_00001": "Q_cfps_max",
    "173616_00060_00001_cd": "Q_cfps_max_status",
    "173617_00060_00002": "Q_cfps_min",
    "173617_00060_00002_cd": "Q_cfps_min_status"
})

# ### Add column that is cms (cubic meters per second)
romayor["Q_cms_mean"] = romayor["Q_cfps_mean"] * 0.0283168
romayor["Q_cms_mean_status"] = romayor["Q_cfps_mean_status"]

## Set the datetime column to be a datetime object
romayor["datetime"] = pd.to_datetime(romayor["datetime"], format='%m/%d/%Y')

# # ### Set the index to the datetime column
romayor = romayor.set_index('datetime')

romayor.tail()

### Plot the full Romayor discharge timeseries (1924-2025) with yearly and monthly mean

In [ ]:
romayor_daily_mean = romayor['Q_cms_mean']
romayor_monthly_mean = romayor['Q_cms_mean'].resample('ME').mean()
romayor_yearly_mean = romayor['Q_cms_mean'].resample('YE').mean()

fig, ax = plt.subplots(figsize=(10,3))
palette = sns.color_palette("viridis", 3)

ax.plot(romayor_daily_mean, color='gray', alpha=0.3, linewidth=0.5, label='Daily Mean Flow')
ax.plot(romayor_monthly_mean, linewidth=1, color=palette[1], label='Monthly Mean Flow')
ax.plot(romayor_yearly_mean, linewidth=1.5, color=palette[0], linestyle='--', label='Yearly Mean Flow')
ax.set_xlabel('Date (Year)', fontsize=14)
ax.set_ylabel(r'Discharge ($m^3/s$)', fontsize=14)
ax.set_xlim(romayor.index.min(), romayor.index.max())
ax.legend(loc='upper right',frameon=True, fontsize=12)
sns.despine(left=False, bottom=False)

plt.tight_layout()
plt.show()

### Plot Qw over the period 2015-2025
This generates plot of Qw included in manuscript Figure 1C.

In [ ]:
# Define the date range for the recent data (2015-2025)
recent_data = romayor.loc['2015-01-01': '2025-12-31']

# Define the figure size and style for the recent data plot
plt.style.use('seaborn-v0_8-whitegrid')
width_mm = 180
height_mm = 40

fig, ax = plt.subplots(figsize=(width_mm/25.4, height_mm/25.4))

# Plot daily mean Qw for the recent data
ax.plot(recent_data.index, recent_data['Q_cms_mean'], color='#1f5081', alpha=1, linewidth=0.75)

# Set axis labels and limits
ax.set_ylabel(r'Daily Mean $Q_w$' + '\n' + r'($m^3\,s^{-1}$)', fontsize=9, color='#333333')

# Clean up x-axis format
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.tick_params(axis='both', which='major', labelsize=8, colors='#333333')

# Add solid black border around the plot
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_color('black')
    ax.spines[spine].set_linewidth(1.2)

# Grid lines (optional adjustment)
ax.grid(axis='y', linestyle='--', alpha=0.25, color='gray')
ax.grid(axis='x', visible=False)

plt.tight_layout()
filepath_png = os.path.join(fig_dir, "Fig_1C.png")
filepath_pdf = os.path.join(fig_dir, "Fig_1C.pdf")
plt.savefig(filepath_png, dpi=500, bbox_inches='tight')
plt.savefig(filepath_pdf, bbox_inches='tight')
plt.show()

### Print discharge stats for 1924-2024 and 2015-2025

In [ ]:
# Print median and 80th percentile for the full period
median_flow_full = romayor['Q_cms_mean'].median()
percentile_90_full = romayor['Q_cms_mean'].quantile(0.90)
print(f'Median Flow (1924-2025): {median_flow_full:.2f} m^3/s')
print(f'90th Percentile Flow (1924-2025): {percentile_90_full:.2f} m^3/s')

# Print median and 80th percentile for the recent period
median_flow_recent = recent_data['Q_cms_mean'].median()
percentile_90_recent = recent_data['Q_cms_mean'].quantile(0.90)
print(f'Median Flow (2015-2025): {median_flow_recent:.2f} m^3/s')
print(f'90th Percentile Flow (2015-2025): {percentile_90_recent:.2f} m^3/s')

### Plot daily mean over period of PlanetScope data availability

In [ ]:
# Define the date range for the recent data (2017-2025)
romayor_2017_2025 = romayor.loc['2017-10-01':'2025-01-01']

fig, ax = plt.subplots(figsize=(width_mm/25.4, height_mm/25.4))

# Plot daily mean Qw for the recent data
ax.plot(romayor_2017_2025.index, romayor_2017_2025['Q_cms_mean'], color='#1f5081', alpha=1, linewidth=0.75)

# Set axis labels and limits
ax.set_ylabel(r'Daily Mean $Q_w$' + '\n' + r'($m^3\,s^{-1}$)', fontsize=9, color='#333333')

# Clean up x-axis format
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.tick_params(axis='both', which='major', labelsize=8, colors='#333333')

# Add solid black border around the plot
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_color('black')
    ax.spines[spine].set_linewidth(1.2)

# Grid lines (optional adjustment)
ax.grid(axis='y', linestyle='--', alpha=0.25, color='gray')
ax.grid(axis='x', visible=False)

plt.tight_layout()
plt.show()

### Compute 80th percentile of flows from the full record and plot the "flood period" for the PlanetScope period (2017-2025)

In [ ]:
p80_FullRecord_Q = romayor['Q_cms_mean'].quantile(0.8)
print(f'80th percentile Q: {p80_FullRecord_Q:.2f} m3/s')

### Count floods from first time 50th percentile until discharge goes below 50th percentile (only if 80th percentile is exceeded)

In [ ]:
import pandas as pd

# Step 1: Calculate percentiles
percentiles = romayor['Q_cms_mean'].quantile([0.5, 0.80])

# Step 2: Determine flood events
flood_events = []
flood_active = False
flood_start_date = None
last_flood_end_date = None

for date, row in romayor_2017_2025.iterrows():
    if not flood_active:
        # Check if the discharge crosses the 50th percentile
        if row['Q_cms_mean'] > percentiles[0.5]:
            flood_start_date = date
            flood_active = True
    else:
        # Check if the discharge drops below the 50th percentile
        if row['Q_cms_mean'] <= percentiles[0.5]:
            # Check if the maximum discharge during the flood period exceeds the 80th percentile
            flood_period_data = romayor_2017_2025.loc[flood_start_date:date]
            if flood_period_data['Q_cms_mean'].max() > percentiles[0.80]:
                # Extend flood event until the last day the discharge remains above the 50th percentile
                flood_end_date = date - pd.Timedelta(days=1)
                # Check if the time since the last flood is less than 14 days
                if last_flood_end_date is not None and (flood_start_date - last_flood_end_date).days < 14:
                    # Modify the end date of the last flood event to be the end date of this flood
                    flood_events[-1] = (flood_events[-1][0], flood_end_date)
                else:
                    flood_events.append((flood_start_date, flood_end_date))
                last_flood_end_date = flood_end_date
            flood_active = False

# If flood is still active at the end of the time series
if flood_active:
    # Check if the maximum discharge during the flood period exceeds the 80th percentile
    flood_period_data = romayor_2017_2025.loc[flood_start_date:]
    if flood_period_data['Q_cms_mean'].max() > percentiles[0.80]:
        flood_end_date = romayor_2017_2025.index[-1]
        if last_flood_end_date is not None and (flood_start_date - last_flood_end_date).days < 14:
            flood_events[-1] = (flood_events[-1][0], flood_end_date)
        else:
            flood_events.append((flood_start_date, flood_end_date))

# Step 3: Store flood start and end dates in a DataFrame
flood_df = pd.DataFrame(flood_events, columns=['Flood Start Date', 'Flood End Date'])
flood_df['Flood Start Discharge'] = flood_df['Flood Start Date'].apply(lambda x: romayor_2017_2025.loc[x]['Q_cms_mean'])
flood_df['Flood End Discharge'] = flood_df['Flood End Date'].apply(lambda x: romayor_2017_2025.loc[x]['Q_cms_mean'])
flood_df['Flood Number'] = range(1, len(flood_df) + 1)

# Step 4: Calculate the flood duration
flood_df['Flood Duration'] = flood_df['Flood End Date'] - flood_df['Flood Start Date']

# Calculate the time since last flood
flood_df['time_since_last_flood'] = flood_df['Flood Start Date'] - flood_df['Flood End Date'].shift(1)

# For the first flood, set the time_since_last_flood as NaN
flood_df.at[0, 'time_since_last_flood'] = pd.NaT
flood_df.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

FLOW_COLOR = sns.color_palette("deep")[0] # A nice blue for the main line
FLOOD_COLOR = 'red'

# Use a flag to ensure the label for the legend is only applied once
flood_label_set = False

# Overlay shaded regions representing flood periods
for _, row in flood_df.iterrows():
    ax.axvspan(
        row['Flood Start Date'],
        row['Flood End Date'],
        color=FLOOD_COLOR,
        alpha=0.15,  # Light shading
        linewidth=0, # No border line
        label='Flood Period' if not flood_label_set else None
    )
    # Set the flag so subsequent spans do not get a label
    flood_label_set = True

# Plot the discharge time series
ax.plot(
    romayor_2017_2025['Q_cms_mean'],
    label='Daily Discharge',
    color=FLOW_COLOR,
    linewidth=0.75,
    zorder=5 # Ensure line is on top of axvspan (which has default zorder=0)
)

# Add a horizontal line for p80_FullRecord_Q
ax.axhline(
    y=p80_FullRecord_Q,
    color='orange',
    linestyle='--',
    linewidth=2,
    label='80th Percentile of Historical Discharge'
)

ax.set_title(
    'River Discharge Time Series with Hydrological Flood Periods',
    fontsize=18,
    fontweight='bold',
    loc='left'
)
ax.set_xlabel('Date (Year)', fontsize=14)
# Use LaTeX formatting for the unit for a professional look
ax.set_ylabel(r'Discharge ($m^3/s$)', fontsize=14) 

# Ensure the X-axis spans the full data range
ax.set_xlim(romayor_2017_2025.index.min(), romayor_2017_2025.index.max())

# Enhance the Legend (now simple since axvspan handles the non-duplicates)
ax.legend(loc='upper left', fontsize=12, frameon=True)

# Final Polish: Remove box spines and ensure tight layout
sns.despine(left=False, bottom=False)
plt.tight_layout()
plt.show()

In [ ]:
PS_dates = [
    datetime(2017, 10, 17), # Start baseflow 1 
    datetime(2018, 1, 13), # End baseflow 1 / Start flood 1
    datetime(2018, 5, 7), # End flood 1 / Start baseflow 2
    datetime(2018, 9, 19), # End baseflow 2 / Start flood 2
    datetime(2019, 9, 13), # End flood 2 / Start baseflow 3
    datetime(2019, 12, 23), # End baseflow 3 / Start flood 3
    datetime(2020, 10, 2), # End flood 3 / Start baseflow 4
    datetime(2020, 11, 30), # End baseflow 4 / Start flood 4
    datetime(2021, 9, 26), # End flood 4 / Start baseflow 5
    datetime(2022, 1, 17), # End baseflow 5 / Start flood 5
    datetime(2022, 6, 19), # End flood 5 / Start baseflow 6
    datetime(2022, 10, 27), # End baseflow 6 / Start flood 6
    datetime(2023, 8, 2), # End flood 6 / Start baseflow 7
    datetime(2023, 12, 25), # End baseflow 7 / Start flood 7
    datetime(2024, 9, 9), # End flood 7 / Start baseflow 8
    datetime(2024, 12, 22) # End baseflow 8
]

dates = pd.DataFrame()
dates['date'] = PS_dates
dates['Q_cms_mean'] = dates['date'].apply(lambda x: romayor.loc[x]['Q_cms_mean'])
dates['Q_cms_mean'] = dates['date'].apply(lambda x: romayor.loc[x]['Q_cms_mean'])
dates.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

FLOW_COLOR = sns.color_palette("deep")[0]
FLOOD_COLOR = 'red'
PS_COLOR = 'black'

flood_label_set = False
ps_label_set = False

# Overlay shaded regions representing flood periods
for _, row in flood_df.iterrows():
    # Only assign the label to the first span to avoid duplicate legend entries
    ax.axvspan(
        row['Flood Start Date'],
        row['Flood End Date'],
        color=FLOOD_COLOR,
        alpha=0.15,  # Light shading
        linewidth=0, # No border line
        label='Flood Period' if not flood_label_set else None,
        zorder=1 # Ensure spans are in the background
    )
    # Set the flag so subsequent spans do not get a label
    flood_label_set = True

# Add vertical lines for each PS Scene date
for date in PS_dates:
    ax.axvline(
        date, 
        color=PS_COLOR, 
        linestyle=':', 
        linewidth=2.0, 
        alpha=0.7, 
        label='PS Scene Acquisition' if not ps_label_set else None,
        zorder=2 # Place above spans but below the main flow line
    )
    ps_label_set = True


# Plot the Q_mean_cms time series LAST so it sits visually on top of the shaded spans and PS lines
ax.plot(
    romayor_2017_2025['Q_cms_mean'],
    label='Daily Discharge',
    color=FLOW_COLOR,
    linewidth=1.5,
    zorder=5 # Ensure line is on top of everything
)

# Add a horizontal line for p80_FullRecord_Q
ax.axhline(
    y=p80_FullRecord_Q,
    color='orange',
    linestyle='--',
    linewidth=2,
    label='80th Percentile of Historical Discharge'
)

ax.set_xlabel('Date (Year)', fontsize=14)
# Use LaTeX formatting for the unit for a professional look
ax.set_ylabel(r'Discharge ($m^3/s$)', fontsize=14) 

# Ensure the X-axis spans the full data range
ax.set_xlim(romayor_2017_2025.index.min(), romayor_2017_2025.index.max())

# Enhance the Legend (all labels are unique due to the flags, so this works cleanly)
ax.legend(loc='upper left', fontsize=12, frameon=True)
sns.despine(left=False, bottom=False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

# Define colors for clarity
FLOW_COLOR = 'black'
PS_COLOR = 'green'
BASEFLOW_SHADE_COLOR = sns.color_palette("pastel")[2]
FLOOD_SHADE_COLOR = sns.color_palette("pastel")[3]

# Use flags to ensure labels for the legend are only applied once
baseflow_label_set = False
flood_shade_label_set = False
ps_label_set = False # Re-initialize the PS label flag

# Iterate between sequential PS dates to shade the period
for i in range(len(PS_dates) - 1):
    start = PS_dates[i]
    end = PS_dates[i+1]
    
    # Determine period type (i=0 is the first period, designated as Baseflow)
    is_baseflow = (i % 2 == 0)
    
    current_color = BASEFLOW_SHADE_COLOR if is_baseflow else FLOOD_SHADE_COLOR
    current_label = 'Baseflow Period' if is_baseflow else 'Flood Period'
    
    # Check if the label needs to be set for the legend
    if is_baseflow and not baseflow_label_set:
        label_to_use = current_label
        baseflow_label_set = True
    elif not is_baseflow and not flood_shade_label_set:
        label_to_use = current_label
        flood_shade_label_set = True
    else:
        label_to_use = None

    # Shade the period
    ax.axvspan(
        start,
        end,
        color=current_color,
        alpha=0.35,
        linewidth=0,
        label=label_to_use,
        zorder=1
    )

# Add vertical lines for each PS Scene date
for date in PS_dates:
    ax.axvline(
        date, 
        color=PS_COLOR, 
        linestyle=':', 
        linewidth=1.0, 
        alpha=0.7, 
        label='PS Scene Acquisition' if not ps_label_set else None,
        zorder=2
    )
    ps_label_set = True


# Plot the Q_mean_cms time series LAST so it sits visually on top of the shaded spans and PS lines
ax.plot(
    romayor_2017_2025['Q_cms_mean'],
    label='Daily Discharge',
    color=FLOW_COLOR,
    linewidth=1.5,
    zorder=5
)

ax.set_xlabel('Date (Year)', fontsize=14)
ax.set_ylabel(r'Discharge ($m^3/s$)', fontsize=14) 

# Ensure the X-axis spans the full data range
ax.set_xlim(romayor_2017_2025.index.min(), romayor_2017_2025.index.max())

# Enhance the Legend (all labels are unique due to the flags, so this works cleanly)
ax.legend(loc='upper left', fontsize=12, frameon=True)
sns.despine(left=False, bottom=False)
plt.tight_layout()
plt.show()

In [ ]:
# Create a new DataFrame to store the periods
new_data = []

for i in range(len(dates) - 1):
    start_date = dates.iloc[i]['date']
    end_date = dates.iloc[i + 1]['date']
    elapsed_time = end_date - start_date
    period = {
        'start_date': start_date,
        'end_date': end_date,
        'duration_days': elapsed_time,
        'Q_cms_mean': dates.iloc[i]['Q_cms_mean'],
    }

    new_data.append(period)
new_df = pd.DataFrame(new_data)
new_df['age'] = range(15)
new_df.head(20)

In [ ]:
start_date = list(new_df['start_date'])
end_date = list(new_df['end_date'])
date_pairs = list(zip(start_date, end_date))
date_pairs

In [ ]:
# Mean values for all periods
mean_Q_per_period = []
for start_date_str, end_date_str in date_pairs:
    start_date = pd.to_datetime(start_date_str)
    end_date = pd.to_datetime(end_date_str)
    filtered_df = romayor.loc[start_date:end_date]
    mean_value = filtered_df['Q_cms_mean'].mean()
    mean_Q_per_period.append(mean_value)

# Max values for all periods
max_Q_per_period = []
for start_date_str, end_date_str in date_pairs:
    start_date = pd.to_datetime(start_date_str)
    end_date = pd.to_datetime(end_date_str)
    filtered_df = romayor.loc[start_date:end_date]
    max_value = filtered_df['Q_cms_mean'].max()
    max_Q_per_period.append(max_value)

### Mean/max values for flood periods
flood_inds = [1,3,5,7,9,11,13]
mean_Q_per_flood = [mean_Q_per_period[ind] for ind in flood_inds]
max_Q_per_flood = [max_Q_per_period[ind] for ind in flood_inds]

### Mean/max values for baseflow periods
baseflow_inds = [0,2,4,6,8,10,12,14]
mean_Q_per_baseflow = [mean_Q_per_period[ind] for ind in baseflow_inds]
max_Q_per_baseflow = [max_Q_per_period[ind] for ind in baseflow_inds]

In [ ]:
### Find the percentile of the historical record that the quiescent periods fall into
from scipy.stats import percentileofscore

# Mean discharge during baseflow periods
baseflow_mean = np.mean(mean_Q_per_baseflow)

# Mean discharge during flood periods
flood_mean = np.mean(mean_Q_per_flood)

# Min discharge at image acquisition
min_discharge = 39.9

# Max discharge at image acquisition
max_discharge = 52.7

# Filter out NaN values from romayor discharge data
valid_data = romayor['Q_cms_mean'].dropna()

# Calculate percentiles for baseflow
baseflow_mean_percentile = percentileofscore(valid_data.values, baseflow_mean)
baseflow_max_percentile = percentileofscore(valid_data.values, np.max(max_Q_per_baseflow))

# Calculate percentiles for flood
flood_mean_percentile = percentileofscore(valid_data.values, np.mean(mean_Q_per_flood))
flood_max_percentile = percentileofscore(valid_data.values, np.max(max_Q_per_flood))

# Calculate percentiles for min and max discharge at image acquisition
min_percentile =  percentileofscore(valid_data.values, min_discharge)
max_percentile =  percentileofscore(valid_data.values, max_discharge)

print(f"Mean daily discharge during baseflow periods was {baseflow_mean:.1f} m3/s which is at the {baseflow_mean_percentile:.1f}th percentile of historical data.")
print(f"Mean daily discharge during flood periods was {flood_mean:.1f} m3/s which is at the {flood_mean_percentile:.1f}th percentile of historical data.")

print(f"The value {min_discharge:.1f} m3/s is at the {min_percentile:.1f}th percentile within the historical data.")
print(f"The value {max_discharge:.1f} m3/s is at the {max_percentile:.1f}th percentile within the historical data.")


### Add cumulative and peak discharge between the two dates

In [ ]:
for index, row in new_df.iterrows():
    start_date = row['start_date']
    end_date = row['end_date']

    # Filter 'romayor' DataFrame based on 'start_date' and 'end_date'
    filtered_romayor = romayor[(romayor.index >= start_date) & (romayor.index <= end_date)]
    
    # Add 'conditions' column to the filtered DataFrame
    if index % 2 == 0:
      new_df.at[index, 'conditions'] = 'B'
    else:
      new_df.at[index, 'conditions'] = 'F'

    # Calculate cumulative discharge for the filtered period
    cumulative_discharge = filtered_romayor['Q_cms_mean'].sum()

  # Calculate peak discharge for each period
    peak_discharge = filtered_romayor['Q_cms_mean'].max()
    # Assign the cumulative and peak discharge value to the corresponding rows in 'new_df'
    new_df.at[index, 'cum_Q_m3'] = cumulative_discharge
    new_df.at[index, 'peak_discharge_m3s'] = peak_discharge

new_df['cum_Q_m3'] = pd.to_numeric(new_df['cum_Q_m3'])
new_df['peak_discharge_m3s'] = pd.to_numeric(new_df['peak_discharge_m3s'])

new_df.head(15)

In [ ]:
table_df = pd.DataFrame({
    "Period #": range(1, len(new_df) + 1),
    "Conditions": new_df["conditions"],
    "Start Date": new_df["start_date"].dt.strftime("%m/%d/%Y"),
    "End Date": new_df["end_date"].dt.strftime("%m/%d/%Y"),
    "Duration (Days)": new_df["duration_days"].dt.days,
    "Cumulative Qw (m3)": new_df["cum_Q_m3"].round(0).astype(int),
    "Peak Qw (m3/s)": new_df["peak_discharge_m3s"].round(0).astype(int),
})

# Drop index
table_df.reset_index(drop=True, inplace=True)

table_df.head(15)

### Write the table to csv
This is manuscript Table 1.

In [ ]:
table_path = table_dir / 'table1.csv'
table_df.to_csv(table_path, index=False)

### Read in bankline polygon graph data

In [ ]:
polygon_graph_path = data_dir / "polygon_graphs" / "polygon_graph.parquet"
polys = gpd.read_parquet(polygon_graph_path)
polys.tail()

### Filter 'noise' polygons
This analysis assumes a combined stage and image coregistration uncertainty of 12 linear meters (4 linear pixels). Distances in the polygon `dist` field are in meters. See manuscript for filtering rationale.

In [ ]:
# Retain only polygons with `dist` values greater than 12 meters 
polys_filt = polys[polys['dist'] > 12].copy()

In [ ]:
### Group the 'signed area' of polygons from polygon graphs by their age
erosion = polys_filt[polys_filt['area_sign'] < 0].groupby('age')
deposition = polys_filt[polys_filt['area_sign'] > 0].groupby('age')

In [ ]:
# Add the total erosional and depositional areas to the discharge dataframe
new_df['da'] = deposition['area_sign'].sum()  #total deposition
new_df['ea'] = abs(erosion['area_sign'].sum()) #total erosion
new_df['ta'] = new_df['da'] + new_df['ea']
new_df['dapd'] = new_df['da'] / new_df['duration_days'].dt.days
new_df['eapd'] = abs(new_df['ea'])/ new_df['duration_days'].dt.days
new_df['tapd'] = new_df['ta'] / new_df['duration_days'].dt.days
new_df.head(20)

### Add discharge values to polygon graph dataframe


In [ ]:
polys = polys.merge(new_df[['age','cum_Q_m3','dapd','eapd','tapd']], on='age', how='left')
polys.head()

### Write polys with discharge to a shapefile 

In [ ]:
polys_with_Q_path = data_dir / "polygon_graphs" / "polygon_graph_w_Q.parquet"
polys.to_parquet(polys_with_Q_path)

### Plot the Qw and kinematic metrics
This generates manuscript Figure 4.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

C_EROS = '#d95f5f'     # Light Red (Pink)
C_DEPO = '#5a8fc8'     # Light Blue

# Global Event Color Dictionary (Baseflows = Even, Floods = Odd)
color_dict = {
    0: '#050C36', 1: '#993404', 2: '#162363', 3: '#c4510b',
    4: '#273A90', 5: '#e57217', 6: '#3851BD', 7: '#fe9929',
    8: '#5F62CD', 9: '#fec46c', 10: '#8673DD', 11: '#fee6a5',
    12: '#AC84ED', 13: '#ffffd4', 14: '#D395FD'
}

# --- Plot 1: Discharge (Dual Axis) ---
ax_cum = ax[0].twinx()

for i, (_, row) in enumerate(new_df.iterrows()):
    ax_cum.fill_between(
        [row['start_date'], row['end_date']], 0, row['cum_Q_m3'], 
        color='lightgrey', alpha=0.5, edgecolor='k', linewidth=0.5, zorder=-1
    )

subset = romayor[(romayor.index > '2017-09-30') & (romayor.index < '2024-12-31')]
ax[0].plot(subset.index, subset['Q_cms_mean'], color='k', zorder=10, linewidth=0.5, label='Mean Daily $Q_w$')

for i, (_, row) in enumerate(new_df.iterrows()):
    label = 'Max $Q_w$' if i == 0 else None
    ax[0].hlines(y=row['peak_discharge_m3s'], xmin=row['start_date'], xmax=row['end_date'],
                 color='red', linestyle='--', linewidth=0.5, zorder=10, label=label)

ax[0].set_zorder(ax_cum.get_zorder() + 1)
ax[0].patch.set_visible(False)
ax[0].legend(loc='upper left', fontsize=8, frameon=True, edgecolor='black', fancybox=False)


# --- Plot 2: Cumulative Bankline ---
total_ta = new_df['ta'].sum()
trans_ax1 = transforms.blended_transform_factory(ax[1].transData, ax[1].transAxes)

for i, (_, row) in enumerate(new_df.iterrows()):
    c = color_dict.get(i, 'grey')
    y_val = row['ta'] * 1e-6
    
    # 1. Plot the main background fill
    ax[1].fill_between([row['start_date'], row['end_date']], 0, y_val, 
                       color=c, alpha=1, edgecolor='k', linewidth=0.5)
    
    x_mid = row['start_date'] + (row['end_date'] - row['start_date']) / 2

    
    if row['ta'] > 0:
        # Add percentage label above each bar 
        pct = round((row['ta'] / total_ta) * 100)
        text_y_pos = y_val + 0.05
        
        ax[1].text(x_mid, text_y_pos, f"{pct}%", 
                   ha='center', va='bottom', fontsize=8.5, color='black')
        
        # Add Event Label (F1, B1, etc.)
        label_text = f"B{(i//2) + 1}" if i % 2 == 0 else f"F{(i//2) + 1}"
        ax[1].text(x_mid, 0.95, label_text, transform=trans_ax1,
                   ha='center', va='top', fontsize=11, fontweight='bold', 
                   color='black', zorder=5)


# --- Plot 3: Erosion / Deposition ---
trans_ax2 = transforms.blended_transform_factory(ax[2].transData, ax[2].transAxes)

for i, (_, row) in enumerate(new_df.iterrows()):
    x_mid = row['start_date'] + (row['end_date'] - row['start_date']) / 2
    
    # Deposition
    if row['da'] > 0:
        pct = round((row['da'] / row['ta']) * 100)
        depo_y = row['da'] * 1e-6
        ax[2].fill_between([row['start_date'], row['end_date']], 0, depo_y, 
                           color=C_DEPO, alpha=0.7, edgecolor='k', linewidth=0.5)
            
        ax[2].text(x_mid, depo_y + 0.05, 
                   f"{pct}%", ha='center', va='bottom', fontsize=8.5, color='black')

    # Erosion
    if row['ea'] > 0:
        pct = round((row['ea'] / row['ta']) * 100)
        eros_y = -row['ea'] * 1e-6
        ax[2].fill_between([row['start_date'], row['end_date']], 0, eros_y, 
                           color=C_EROS, alpha=0.7, edgecolor='k', linewidth=0.5)
        ax[2].text(x_mid, eros_y - 0.05, 
                   f"{pct}%", ha='center', va='top', fontsize=8.5, color='black')

    # Add Event Label
    label_text = f"B{(i//2) + 1}" if i % 2 == 0 else f"F{(i//2) + 1}"
    ax[2].text(x_mid, 0.95, label_text, transform=trans_ax2,
               ha='center', va='top', fontsize=11, fontweight='bold', 
               color='black', zorder=5)


# --- Vertical Event Dividers ---
for _, row in dates.iterrows():
    for axis in ax:
        axis.axvline(x=row['date'], color='black', linestyle='--', linewidth=0.8, alpha=0.75, zorder=0)


# --- Formatting & Limits ---
ax[0].set_xlim([datetime(2017, 9, 30), datetime(2024, 12, 31)])
ax[0].set_ylim(0, 5000)
ax_cum.set_ylim(0, 3e5)

# Limits calculated strictly from the base bar height since error is downward
max_ta = (new_df['ta'] * 1e-6).max()
ax[1].set_ylim(0, max_ta * 1.35) 

ax[2].set_ylim(-1, 1.35)

# Axis labels
ax[2].set_xlabel('Date', fontsize=12)
ax[0].set_ylabel(r'Mean Daily $Q_w$ (m$^3\,\mathrm{s}^{-1}$)', fontsize=11)
ax_cum.set_ylabel(r'Cumulative $Q_w$ (m$^3$)', fontsize=11, color='dimgrey')
ax[1].set_ylabel(r'Cumulative Bankline Change (km$^2$)', fontsize=11)
ax[2].set_ylabel(r'Erosion / Deposition (km$^2$)', fontsize=11)

ax_cum.tick_params(axis='y', colors='dimgrey')
ax_cum.spines['right'].set_color('dimgrey')

ax[2].fill_between([], [], color=C_DEPO, alpha=1.0, linewidth=0.5, linestyle='-', label='Deposition')
ax[2].fill_between([], [], color=C_EROS, alpha=1.0, linewidth=0.5, linestyle='-', label='Erosion')
ax[2].legend(loc='lower left', fontsize=8, frameon=True, edgecolor='black', fancybox=True)


# --- Custom Scientific Notation ---
ax[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: '{:.1f}'.format(x / 1000)))
ax[0].text(0.0, 1.01, r'$\times 10^3$', transform=ax[0].transAxes, ha='left', va='bottom', fontsize=10)

ax_cum.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: '{:.0f}'.format(x / 1000)))
ax_cum.text(1.0, 1.01, r'$\times 10^3$', transform=ax_cum.transAxes, ha='right', va='bottom', fontsize=10, color='dimgrey')

for axis in ax:
    axis.tick_params(axis='both', which='major', direction='out', length=4, width=0.8, 
                     colors='k', left=True, bottom=True, labelsize=10, zorder=10)
    axis.xaxis.set_minor_locator(mdates.MonthLocator())
    axis.tick_params(axis='x', which='minor', direction='out', length=2, width=0.6, 
                     colors='k', bottom=True, labelsize=0)
    axis.grid(False)

ax_cum.tick_params(axis='y', which='major', direction='out', length=4, width=0.8, 
                   colors='dimgrey', right=True, labelsize=10)
ax_cum.grid(False)

fig.align_ylabels(ax[:]) 
plt.tight_layout()
filepath_png = os.path.join(fig_dir, "Fig_4.png")
filepath_pdf = os.path.join(fig_dir, "Fig_4.pdf")
plt.savefig(filepath_png, dpi=500, bbox_inches='tight')
plt.savefig(filepath_pdf, bbox_inches='tight')
plt.show()

### Scatter plot Qw vs bankline change
This generates manuscript Figure 5.

In [ ]:
# Set Seaborn aesthetics
sns.set_theme(style="ticks", context="notebook", font_scale=1.1)

# Initialize the 1x2 Figure
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)

# Iterate through the dataframe and plot point-by-point
for i, (_, row) in enumerate(new_df.iterrows()):
    
    # Determine event type, label, and marker shape
    is_baseflow = (i % 2 == 0)
    label_text = f"F{(i//2) + 1}" if not is_baseflow else f"B{(i//2) + 1}"
    marker = '^' if is_baseflow else 'o'
    c = color_dict.get(i, 'grey')
    
    # Extract values (converting area to km^2 just like Fig 4)
    y_val = row['ta'] * 1e-6
    x_peak = row['peak_discharge_m3s']
    x_cum = row['cum_Q_m3']
    
    # --- Plot Panel A (Peak Discharge) ---
    axes[0].scatter(x_peak, y_val, color=c, marker=marker, 
                    s=120, edgecolor='black', linewidth=0.5, zorder=5)
    
    # Add text label ONLY if it is a flood event
    if not is_baseflow:
        axes[0].annotate(label_text, (x_peak, y_val), 
                         textcoords="offset points", xytext=(8, 4), 
                         fontsize=9.5, fontweight='bold', zorder=6)
    
    # --- Plot Panel B (Cumulative Discharge) ---
    axes[1].scatter(x_cum, y_val, color=c, marker=marker, 
                    s=120, edgecolor='black', linewidth=0.5, zorder=5)
    
    # Add text label ONLY if it is a flood event
    if not is_baseflow:
        axes[1].annotate(label_text, (x_cum, y_val), 
                         textcoords="offset points", xytext=(8, 4), 
                         fontsize=9.5, fontweight='bold', zorder=6)

# Format Axes and Labels
axes[0].set_xlabel(r'Peak Discharge (m$^3\,\mathrm{s}^{-1}$)', fontweight='bold', labelpad=10)
axes[1].set_xlabel(r'Cumulative Discharge (m$^3$)', fontweight='bold', labelpad=10)
axes[0].set_ylabel(r'Total Bankline Change (km$^2$)', fontweight='bold', labelpad=10)
max_peak = new_df['peak_discharge_m3s'].max()
axes[0].set_xlim(right=max_peak * 1.15)
max_cum = new_df['cum_Q_m3'].max()
axes[1].set_xlim(right=max_cum * 1.15)

# Tick Formatting (Standard comma format for both axes)
axes[0].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: '{:,.0f}'.format(x)))
axes[1].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: '{:,.0f}'.format(x)))

# Clean up gridlines and ensure solid axes boundaries
for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5, zorder=0)
    # Ensure all spines are visible to create a solid box
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(0.8)

# Create a Custom Legend for Marker Types (Flood vs Baseflow)
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Flood (F)', 
           markerfacecolor='dimgrey', markeredgecolor='black', markeredgewidth=0.5, markersize=10),
    Line2D([0], [0], marker='^', color='w', label='Baseflow (B)', 
           markerfacecolor='dimgrey', markeredgecolor='black', markeredgewidth=0.5, markersize=10)
]
axes[0].legend(handles=legend_elements, loc='upper left', frameon=True, 
               edgecolor='black', fancybox=False, fontsize=10)

# Add Subplot Letter Labels (A & B)
axes[0].text(-0.15, 1.05, '(A)', transform=axes[0].transAxes, fontsize=14, fontweight='bold', va='top')
axes[1].text(-0.05, 1.05, '(B)', transform=axes[1].transAxes, fontsize=14, fontweight='bold', va='top')

plt.tight_layout()
filepath_png = os.path.join(fig_dir, "Fig_5.png")
filepath_pdf = os.path.join(fig_dir, "Fig_5.pdf")
plt.savefig(filepath_png, dpi=500, bbox_inches='tight')
plt.savefig(filepath_pdf, bbox_inches='tight')
plt.show()

## Segment and analyze bend-scale kinematics 
For temporally-consistent bend segementation we use a Landsat-derived centerline generated previously by Speed et al (2024). This centerline is used to identify bend inflection points and segment the river reach into discrete bends.

### Load the 2017 Landsat centerline 

In [ ]:
# Read the 2017 Landsat centerline that has been trimmed to our AOI
cl_2017 = gpd.read_file(data_dir / 'centerlines' / 'landsat' / 'cline_LC08_025039_20170929_clp.geojson')
geom = cl_2017.geometry.iloc[0]
x = np.array(geom.xy[0])
y = np.array(geom.xy[1])

### Apply gentle smoothing

In [ ]:
savgol_window = 31
savgol_poly_order = 3
x = savgol_filter(x, savgol_window, savgol_poly_order)
y = savgol_filter(y, savgol_window, savgol_poly_order)

### Compute curvature and find inflection points

In [ ]:
# Compute curvature and find zero crossings
curv, s = rb.compute_curvature(x, y)
loc_zero_curv, loc_max_curv = rb.find_zero_crossings(curv)
x_inf = x[loc_zero_curv]
y_inf = y[loc_zero_curv]

# Apply a distance filter to ensure points are at least 125 m apart
if len(x_inf) > 0:
    filtered_x = [x_inf[0]]
    filtered_y = [y_inf[0]]

    for i in range(1, len(x_inf)):
        # Calculate Euclidean distance to the LAST KEPT point
        dist = np.sqrt((x_inf[i] - filtered_x[-1])**2 + (y_inf[i] - filtered_y[-1])**2)
        
        # Only keep the point if it is >= 125 away from the previous valid point
        if dist >= 250:
            filtered_x.append(x_inf[i])
            filtered_y.append(y_inf[i])

    # Overwrite the original arrays with the filtered data
    x_inf = np.array(filtered_x)
    y_inf = np.array(filtered_y)

### Write inflection points to a file

In [ ]:
# Create a GeoDataFrame
inf_geometry = [Point(x, y) for x, y in zip(x_inf, y_inf)]
gdf_inf = gpd.GeoDataFrame(geometry=inf_geometry, crs=cl_2017.crs)

# Save GeoDataFrame to a geojson
gdf_inf.to_file(data_dir / 'centerlines' / 'landsat' / 'cline_LC08_025039_20170929_clp_inf_pts.geojson', driver='GeoJSON')

# Read the data to a variable
inf_points = gpd.read_file(data_dir / 'centerlines' / 'landsat' / 'cline_LC08_025039_20170929_clp_inf_pts.geojson')
inf_points.head(10)

### Segment the 2017 PlanetScope centerline using the Landsat-derived inflection points

In [ ]:
# Load the smoothed centerline for the 2017 Planetscope scene
cl_planet = gpd.read_file(data_dir / 'centerlines' / 'planet' / '20171017_DDWI_binary_m1_smoothed_cl.geojson')
planet_line = cl_planet.geometry.iloc[0]
planet_len = planet_line.length

# --- Filter Inflection Points ---
valid_split_distances = []
MAX_OFFSET = 200

for pt in inf_points.geometry:
    # Filter points that are too far or clamp to the start
    if planet_line.distance(pt) > MAX_OFFSET:
        continue
        
    dist_along = planet_line.project(pt)
    
    # Trim Upstream Tail (keep points > 1.0m)
    # Keep End Clamping (points at the end are valid delimiters)
    if dist_along > 1.0:
        valid_split_distances.append(dist_along)

# Define Cut Points (Start of Bend 1 -> End of Last Bend)
unique_cuts = set(valid_split_distances)
unique_cuts.add(planet_len) 
cuts = sorted(list(unique_cuts))

# --- Segment Centerline & Generate Separators ---
segments = []
separators = []
half_len = 100  # Visual length of separator lines (400m total)

# Create Segments
for i in range(len(cuts) - 1):
    start_dist = cuts[i]
    end_dist = cuts[i+1]
    
    if (end_dist - start_dist) > 0.001: 
        segment = substring(planet_line, start_dist, end_dist)
        segments.append(segment)

# Create Separators at every cut location
for dist in cuts:
    # Interpolate point on line
    point = planet_line.interpolate(dist)
    
    # Calculate local tangent
    d_back = max(0, dist - 1.0)
    d_fwd = min(planet_len, dist + 1.0)
    p_back = planet_line.interpolate(d_back)
    p_fwd = planet_line.interpolate(d_fwd)
    
    dx = p_fwd.x - p_back.x
    dy = p_fwd.y - p_back.y
    angle = np.arctan2(dy, dx)
    
    # Create and rotate perpendicular line
    vertical_line = LineString([(0, -half_len), (0, half_len)])
    rotated_line = rotate(vertical_line, angle, origin=(0, 0), use_radians=True)
    final_sep = translate(rotated_line, xoff=point.x, yoff=point.y)
    separators.append(final_sep)

# Save Segments (IDs start at 1)
bend_ids = range(1, len(segments) + 1)
gdf_segments = gpd.GeoDataFrame({'bend_id': bend_ids, 'geometry': segments}, crs=cl_planet.crs)
gdf_segments.to_file(data_dir / 'centerlines' / 'planet' / 'cline_PS_20171017_split.geojson')

# Save Separators
gdf_separators = gpd.GeoDataFrame(geometry=separators, crs=cl_planet.crs)
gdf_separators.to_file(data_dir / 'centerlines' / 'planet' / 'cline_PS_20171017_separators.geojson')

print(f"Created {len(gdf_segments)} segments and {len(gdf_separators)} separators.")

# --- Voronoi Partitioning ---
points = []
ids = []

# Densify segments for Voronoi seeding
for idx, row in gdf_segments.iterrows():
    length = row.geometry.length
    num_points = int(length / 10) + 1 
    
    for i in range(num_points):
        pt = row.geometry.interpolate(i * 10)
        points.append(pt)
        ids.append(row['bend_id'])

gdf_points = gpd.GeoDataFrame({'bend_id': ids, 'geometry': points}, crs=gdf_segments.crs)

# Generate Regions
envelope = gdf_segments.unary_union.envelope.buffer(1000) 
regions = voronoi_diagram(MultiPoint(gdf_points.geometry.tolist()), envelope=envelope)

# Attribute and Dissolve
voronoi_gdf = gpd.GeoDataFrame(geometry=list(regions.geoms), crs=gdf_segments.crs)
voronoi_attributed = gpd.sjoin(voronoi_gdf, gdf_points, predicate='contains')
bends_partition = voronoi_attributed.dissolve(by='bend_id')

# Clip to extent
river_buffer_mask = gdf_segments.unary_union.buffer(300) 
bends_partition = bends_partition.clip(river_buffer_mask)

#Sort by bend_id for clarity
bends_partition = bends_partition.sort_values(by='bend_id')

bends_partition.to_file(data_dir / 'centerlines' / 'planet' / 'bends_partition_PS_20171017.geojson')

### Assign polygons to the newly segmented bends

In [ ]:
# --- Polygon Assignment ---
polygon_graph_path = data_dir / 'polygon_graphs' / 'polygon_graph.parquet'
if polys.crs != bends_partition.crs:
    polys = polys.to_crs(bends_partition.crs)

polys['centroid_geom'] = polys.geometry.centroid
polys_centroids = polys.set_geometry('centroid_geom')

polygon_graph_by_bend_dir = data_dir / "polygon_graphs" / 'polygon_graphs_by_bend'
polygon_graph_by_bend_dir.mkdir(parents=True, exist_ok=True)

print(f"Assigning polygons to {len(bends_partition)} bends...")

for bend_id, row in bends_partition.iterrows():
    # Spatial filter using centroids
    mask = polys_centroids.within(row.geometry)
    current_bend_polys = polys.loc[mask].copy()
    
    if len(current_bend_polys) == 0:
        continue
        
    current_bend_polys['bend'] = bend_id
    
    # Cleanup and Save
    if 'centroid_geom' in current_bend_polys.columns:
        current_bend_polys = current_bend_polys.drop(columns=['centroid_geom'])
        
    outfile = polygon_graph_by_bend_dir / f'bend_{bend_id}.parquet'
    current_bend_polys.to_parquet(outfile)

print("Complete.")

### Segment the polygon graphs by age

In [ ]:
# Define and create the output directory using pathlib
polygon_graph_by_age_dir = data_dir / "polygon_graphs" / 'polygon_graphs_by_age'
polygon_graph_by_age_dir.mkdir(parents=True, exist_ok=True)

# Group by the 'age' column
grouped = polys.groupby('age')

# Iterate over groups and export each to individual GeoParquet files
for age, group in grouped:
    # Find all geometry columns in the GeoDataFrame
    geom_cols = group.select_dtypes(include=['geometry']).columns
    
    # Identify which ones are NOT the active geometry and drop them
    cols_to_drop = [col for col in geom_cols if col != group.geometry.name]
    clean_group = group.drop(columns=cols_to_drop)
    
    # Define the filepath using pathlib and export
    filename = polygon_graph_by_age_dir / f'polygon_graph_age_{age}.parquet'
    clean_group.to_parquet(filename)

### Read in the polygon graphs segmented by bend (if not already in memory)

In [ ]:
# Define a function to extract the numeric part of the filenames
def extract_num(filename):
    return int(os.path.basename(filename).split('_')[1].split('.')[0])

bends = glob(os.path.join(polygon_graph_by_bend_dir, '*.parquet'))
bends_sorted = sorted(bends, key=extract_num)

bend_dfs = []
for bend in bends_sorted:
    bend_dfs.append(gpd.read_parquet(bend))
print(f"Loaded {len(bend_dfs)} bend GeoParquet files into memory.")

### Aggregate bend information

In [ ]:
areas_per_age = []
depo_eros_per_age = []

ALL_AGES = np.arange(15)

for bend in bend_dfs:

    areas = []
    depo_eros = []

    for age in ALL_AGES:
        age_group = bend[bend['age'] == age]

        # geometry areas
        area_values = age_group.geometry.area.values
        area_values = area_values[area_values > 81]

        # signed erosion / deposition
        depo_eros_values = age_group['area_sign'].values
        depo_eros_values = depo_eros_values[np.abs(depo_eros_values) > 81]

        # ALWAYS append something (possibly empty)
        areas.append(area_values)
        depo_eros.append(depo_eros_values)

    areas_per_age.append(areas)
    depo_eros_per_age.append(depo_eros)

areas_per_age = np.array(areas_per_age, dtype='object')
depo_eros_per_age = np.array(depo_eros_per_age, dtype='object')

In [ ]:
n_bends = areas_per_age.shape[0]
n_ages = areas_per_age.shape[1]

# Initialize destination arrays (Ages, Bends) for easy plotting later
total_change = np.zeros((n_ages, n_bends))
erosion = np.zeros((n_ages, n_bends))
deposition = np.zeros((n_ages, n_bends))

# Iterate and Aggregate
for bend_idx in range(n_bends):
    for age_idx in range(n_ages):
        
        # NOTICE: We access areas_per_age as [bend, age]
        ag_data = areas_per_age[bend_idx][age_idx]
        
        # --- Total Change ---
        if len(ag_data) > 0:
            # Store in total_change as [age, bend] for plotting convenience
            total_change[age_idx, bend_idx] = np.sum(ag_data)
            
        # --- Erosion / Deposition ---
        de_data = depo_eros_per_age[bend_idx][age_idx]
        if len(de_data) > 0:
            # Sum positive values (Deposition)
            deposition[age_idx, bend_idx] = np.sum(de_data[de_data > 0])
            # Sum negative values (Erosion)
            erosion[age_idx, bend_idx] = np.sum(de_data[de_data < 0])

print("Aggregation complete.")
print(f"Bends: {n_bends}, Ages: {n_ages}")

### Plot the streamwise kinematics
This cell generates manuscript Figure 6.

In [ ]:
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# --- PLOTTING CONFIGURATION ---
N_ROWS = 5
N_COLS = 3
TOTAL_PLOTS = N_ROWS * N_COLS

# --- VISUAL CONSTANTS (Meters) ---
# Centered Y-limits to handle negative erosion and positive deposition
Y_LIM = (-35000, 35000) 

# Colors
C_EROS = '#d95f5f'     # Light Red (Pink)
C_DEPO = '#5a8fc8'     # Light Blue
BG_GRAY = '#E0E0E0'    # Gray background
BG_WHITE = 'white'     # White background

fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=(12, 6), 
    sharex=True,
    sharey=True
)

ax_flat = axes.flatten()
x_vals = np.arange(n_bends)

for i in range(TOTAL_PLOTS):
    ax = ax_flat[i]
    
    if i >= n_ages: # Assuming n_ages is 15
        ax.axis('off')
        continue

    # --- 1. ALTERNATING BACKGROUNDS ---
    if i % 2 == 0:
        ax.set_facecolor(BG_WHITE)
    else:
        ax.set_facecolor(BG_GRAY)
    
    ax.grid(False) 

    row = i // N_COLS
    col = i % N_COLS

    # --- 2. PLOT PRIMARY DATA (Erosion & Deposition) ---
    ax.plot(x_vals, erosion[i], '.', color=C_EROS, ms=2)
    ax.plot(x_vals, deposition[i], '.', color=C_DEPO, ms=2)

    # --- SMOOTHED LINES & FILL ---
    win = min(7, n_bends // 2 * 2 + 1)
    if win > 3: 
        eros_s = savgol_filter(erosion[i], win, 3)
        depo_s = savgol_filter(deposition[i], win, 3)
        
        # Fill between the smoothed line and the 0 axis
        ax.fill_between(x_vals, eros_s, 0, color=C_EROS, alpha=0.5, linewidth=0)
        ax.fill_between(x_vals, depo_s, 0, color=C_DEPO, alpha=0.5, linewidth=0)

    # Dashed Zero Line
    ax.axhline(0, color='k', ls='--', lw=0.8)

    # --- 3. ADD HYDRODYNAMIC ZONES ---
    ax.axvline(x=50, color='gray', linestyle='--', linewidth=1)
    ax.axvline(x=70, color='gray', linestyle='--', linewidth=1)

    if i == 0:
        # Shifted annotations down slightly to match the new Y_LIM
        arrow_y = 28000
        text_y = 23000 
        
        ax.annotate('', xy=(0, arrow_y), xytext=(50, arrow_y), 
                       arrowprops=dict(arrowstyle='<->', ls='--', lw=1.0, color='black'), zorder=4)
        ax.text(25, text_y, 'Quasi-Uniform', ha='center', va='top', fontsize=7, fontweight='bold', zorder=5)

        ax.annotate('', xy=(50, arrow_y), xytext=(70, arrow_y), 
                       arrowprops=dict(arrowstyle='<->', ls='--', lw=1.0, color='black'), zorder=4)
        ax.text(60, text_y, 'Transitional', ha='center', va='top', fontsize=7, fontweight='bold', zorder=5)

        ax.annotate('', xy=(70, arrow_y), xytext=(n_bends, arrow_y), 
                       arrowprops=dict(arrowstyle='<->', ls='--', lw=1.0, color='black'), zorder=4)
        midpt_bw = 70 + (n_bends - 70) / 2
        ax.text(midpt_bw, text_y, 'Backwater', ha='center', va='top', fontsize=7, fontweight='bold', zorder=5)

    # --- 4. AXIS HANDLING ---
    ax.set_ylim(Y_LIM)
    ax.set_xlim(0, n_bends + 1)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))

    # Single y-axis ticks
    tick_locs = [-25000, 0, 25000]
    ax.yaxis.set_major_locator(ticker.FixedLocator(tick_locs))

    ax.tick_params(
        axis='both', which='major', direction='out', length=3, width=1, 
        color='black', bottom=True, left=True, top=False, right=False, labelsize=10
    )

    # --- CORRECTED EPISODE NUMBERING (B1, F1, B2, F2...) ---
    ep_type = 'B' if i % 2 == 0 else 'F'
    ep_num = (i // 2) + 1
    panel_label = f"{ep_type}{ep_num}"

    ax.text(0.95, 0.08, panel_label, transform=ax.transAxes, 
            ha='right', va='bottom', fontweight='bold', fontsize=10)

    # --- 5. CLEANUP ---
    if row < N_ROWS - 1:
        ax.tick_params(labelbottom=False)
    if col != 0:
        ax.tick_params(labelleft=False)

# --- GLOBAL LABELS ---
fig.supxlabel(r"Bend #", fontsize=12, y=0.04) 

# Shifted the dual-colored labels to the left side since it is now the primary axis
fig.text(0.06, 0.35, r"(-) Erosion (m$^2$)", 
         va='center', ha='center', rotation=90, fontsize=12, 
         color=C_EROS, fontweight='bold')

fig.text(0.06, 0.50, "/", 
         va='center', ha='center', rotation=90, fontsize=14, 
         color='black')

fig.text(0.06, 0.65, r"(+) Deposition (m$^2$)", 
         va='center', ha='center', rotation=90, fontsize=12, 
         color=C_DEPO, fontweight='bold')

plt.subplots_adjust(top=0.95, bottom=0.12, left=0.12, right=0.95, wspace=0.05, hspace=0.2)

fig_path_png = os.path.join(fig_dir, "Fig_6.png")
fig_path_pdf = os.path.join(fig_dir, "Fig_6.pdf")
fig.savefig(fig_path_png, dpi=500, bbox_inches='tight')
fig.savefig(fig_path_pdf, dpi=500, bbox_inches='tight')

plt.show()

## Analyze reach- and bend-scale bar polygon preservation

### Load polygon graph data 

In [ ]:
polygon_graphs = gpd.read_parquet(data_dir / 'polygon_graphs' / 'polygon_graph.parquet')
polygon_graphs.head()

### Load bar polygon graph data

In [ ]:
bar_polys = gpd.read_parquet(data_dir / 'bar_graphs' / 'bar_polys.parquet')
bar_polys.head()

### Segment bar polygon graphs by bend

In [ ]:
bar_polygon_graph_by_bend_dir = data_dir / 'bar_graphs' / 'bar_polygon_graphs_by_bend'
bar_polygon_graph_by_bend_dir.mkdir(parents=True, exist_ok=True)

print(f"Assigning polygons to {len(bends_partition)} bends...")

bar_polys['centroid_geom'] = bar_polys.geometry.centroid
bar_polys_centroids = bar_polys.set_geometry('centroid_geom')

for bend_id, row in bends_partition.iterrows():
    
    # Spatial filter using centroids
    mask = bar_polys_centroids.within(row.geometry)
    current_bend_polys = bar_polys.loc[mask].copy()
    
    if len(current_bend_polys) == 0:
        continue
        
    current_bend_polys['bend'] = bend_id
    
    # Cleanup and Save
    if 'centroid_geom' in current_bend_polys.columns:
        current_bend_polys = current_bend_polys.drop(columns=['centroid_geom'])
        
    outfile = bar_polygon_graph_by_bend_dir / f'bend_{bend_id}.parquet'
    current_bend_polys.to_parquet(outfile)

print("Complete.")

### Segment bar polygon graphs by age

In [ ]:
bar_polygon_graph_by_age_dir = data_dir / 'bar_graphs' / 'bar_polygon_graphs_by_age'
bar_polygon_graph_by_age_dir.mkdir(parents=True, exist_ok=True)

# Group by the 'age' column
grouped = bar_polys.groupby('age')

# Iterate over groups and export each group to individual GeoParquet files
for age, group in grouped:
    # Define the filename 
    filename = bar_polygon_graph_by_age_dir / f'bar_polygon_graph_age_{age}.parquet'
    
    # Cleanup: Drop 'centroid_geom' if it exists
    if 'centroid_geom' in group.columns:
        group = group.drop(columns=['centroid_geom'])

    # Export the group
    group.to_parquet(filename)

### Read in the polygon graphs segmented by bend

In [ ]:
bends_dir = data_dir / 'polygon_graphs' / 'polygon_graphs_by_bend'
bends = glob(os.path.join(bends_dir, '*.parquet'))
bends_sorted = sorted(bends, key=extract_num)

polygon_graphs_bends_dfs = []
for bend in bends_sorted:
    polygon_graphs_bends_dfs.append(gpd.read_parquet(bend))
print(len(polygon_graphs_bends_dfs), "bends loaded from polygon_graphs_by_bend.")

### Read in the bar polygon graphs segmented by bend

In [ ]:
bends_dir = data_dir / 'bar_graphs' / 'bar_polygon_graphs_by_bend'
bends = glob(os.path.join(bends_dir, '*.parquet'))
bends_sorted = sorted(bends, key=extract_num)

bar_poly_bend_dfs = []
for bend in bends_sorted:
    bar_poly_bend_dfs.append(gpd.read_parquet(bend))
print(len(bar_poly_bend_dfs), "bends loaded from bar_polygon_graphs_by_bend.")

### Plot spatial and temporal preservation data
This cell aggregates the polygon graph and bar graph data to compute preservation per bend and per event. This information is displayed in manuscript Figure 12.

In [ ]:
# Define functions to extract numeric identifiers from filenames for sorting
def extract_num_bend(filename):
    return int(os.path.basename(filename).split('_')[1].split('.')[0])

def extract_num_age(filename):
    return int(os.path.basename(filename).split('_')[-1].split('.')[0])

# Define directories for polygon and bar graphs by bend and age
bends_dir_poly = data_dir / 'polygon_graphs' / 'polygon_graphs_by_bend'
age_dir_poly = data_dir / 'polygon_graphs' / 'polygon_graphs_by_age'
bar_dir_bend = data_dir / 'bar_graphs' / 'bar_polygon_graphs_by_bend'
bar_dir_age = data_dir / 'bar_graphs' / 'bar_polygon_graphs_by_age'

# Load GeoParquet files into GeoDataFrames, sorted by bend or age
polygon_graphs_bends_dfs = [gpd.read_parquet(f) for f in sorted(glob(os.path.join(bends_dir_poly, '*.parquet')), key=extract_num_bend)]
bar_poly_bend_dfs = [gpd.read_parquet(f) for f in sorted(glob(os.path.join(bar_dir_bend, '*.parquet')), key=extract_num_bend)]
polygon_graphs_age_dfs = [gpd.read_parquet(f) for f in sorted(glob(os.path.join(age_dir_poly, '*.parquet')), key=extract_num_age)]
bar_polys_by_age_dfs = [gpd.read_parquet(f) for f in sorted(glob(os.path.join(bar_dir_age, '*.parquet')), key=extract_num_age)]

print("Data loaded. Generating filtered bubble visualization plot...")

NOISE_THRESHOLD_M = 12

# ==========================================
# 1. PROCESS BY BEND (FILTERED)
# ==========================================
depositional_area_per_bend = []
for df in polygon_graphs_bends_dfs:
    valid_depo = df[(df['area_sign'] > 0) & (df['dist'].abs() > NOISE_THRESHOLD_M)]
    depositional_area_per_bend.append(valid_depo.geometry.area.sum())
depositional_area_per_bend = np.array(depositional_area_per_bend)

bar_poly_area_per_bend = []
for df in bar_poly_bend_dfs:
    valid_bars = df[df['dist'].abs() > NOISE_THRESHOLD_M]
    bar_poly_area_per_bend.append(valid_bars.geometry.area.sum())
bar_poly_area_per_bend = np.array(bar_poly_area_per_bend)

with np.errstate(divide='ignore', invalid='ignore'):
    preservation_per_bend = np.nan_to_num(np.divide(bar_poly_area_per_bend, depositional_area_per_bend), nan=0) * 100

# ==========================================
# 2. PROCESS BY AGE (FILTERED)
# ==========================================
depositional_area_per_age = []
for df in polygon_graphs_age_dfs:
    valid_depo = df[(df['area_sign'] > 0) & (df['dist'].abs() > NOISE_THRESHOLD_M)]
    depositional_area_per_age.append(valid_depo.geometry.area.sum())
depositional_area_per_age = np.array(depositional_area_per_age)

bar_poly_area_per_age = []
for df in bar_polys_by_age_dfs:
    valid_bars = df[df['dist'].abs() > NOISE_THRESHOLD_M]
    bar_poly_area_per_age.append(valid_bars.geometry.area.sum())
bar_poly_area_per_age = np.array(bar_poly_area_per_age)

with np.errstate(divide='ignore', invalid='ignore'):
    preservation_per_age = np.nan_to_num(np.divide(bar_poly_area_per_age, depositional_area_per_age), nan=0) * 100

sum_of_preserved_area = np.sum(bar_poly_area_per_age)
areas = (bar_poly_area_per_age / sum_of_preserved_area) * 100

# ==========================================
# 3. PLOTTING AESTHETICS (BUBBLE VIZ)
# ==========================================
with plt.rc_context({'font.size': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8}):
    
    # --- Create Custom X-Axis Labels (B1, F1, B2, F2...) ---
    event_labels = []
    for i in range(len(areas)):
        if i % 2 == 0:
            event_labels.append(f"B{(i//2) + 1}") # Even: 0->B1, 2->B2
        else:
            event_labels.append(f"F{(i//2) + 1}") # Odd: 1->F1, 3->F2
            
    fig, ax = plt.subplots(3, 1, figsize=(5.5, 6.5)) 

    color_dict = {
        0: '#050C36', 1: '#993404', 2: '#162363', 3: '#c4510b',
        4: '#273A90', 5: '#e57217', 6: '#3851BD', 7: '#fe9929',
        8: '#5F62CD', 9: '#fec46c', 10: '#8673DD', 11: '#fee6a5',
        12: '#AC84ED', 13: '#ffffd4', 14: '#D395FD'
    }
    custom_colors = [color_dict[i] for i in range(len(areas))]

    for axis in ax:
        for spine in axis.spines.values():
            spine.set_visible(True)
            spine.set_color('black')
            spine.set_linewidth(1.2)
        axis.grid(True, axis='y', linestyle='--', alpha=0.5, color='gray')
        axis.grid(False, axis='x')
        axis.set_axisbelow(True)
        
    ax[0].grid(False)

    # --- Subplot 1: Preservation per Bend (Bubble Viz) ---
    num_bends = len(preservation_per_bend)
    x_bends = np.arange(num_bends)

    # Define lengths for each hydrodynamic section
    length_quasi = 53.0
    length_trans = 20.0
    length_back = 19.0
    
    sections = [
        ("Quasi-Uniform", 0, 50, length_quasi),
        ("Transitional", 50, 70, length_trans),
        ("Backwater", 70, num_bends, length_back)
    ]

    max_area = np.max(depositional_area_per_bend) if np.max(depositional_area_per_bend) > 0 else 1
    sizes = (depositional_area_per_bend / max_area) * 250 
    ax[0].scatter(x_bends, preservation_per_bend, s=sizes, color='black', alpha=0.5, edgecolor='black', linewidth=0.5, zorder=3)
        
    ax[0].set_xlabel('Bend #') 
    ax[0].set_ylabel('Preservation per bend (%)')
    ax[0].set_ylim(0, 110)

    # Setup secondary y-axis for Subplot 1
    ax0_twin = ax[0].twinx()
    ax0_twin.grid(False)
    ax0_twin.patch.set_visible(False)
    
    ax0_twin.set_ylabel('Total Preserved (% / km)', color='#B23A48')
    ax0_twin.tick_params(axis='y', labelcolor='#B23A48', labelsize=8)
    
    for spine in ax0_twin.spines.values():
        spine.set_visible(False)
    ax0_twin.spines['right'].set_visible(True)
    ax0_twin.spines['right'].set_color('#B23A48')
    ax0_twin.spines['right'].set_linewidth(1.2)

    max_sec_perc = 0

    # Calculate and plot section metrics
    for name, start, end, sec_length in sections:
        # 1. RED LINE: Normalized % of total preserved area (Right Axis)
        pres_sec = np.sum(bar_poly_area_per_bend[start:end])
        sec_perc = (pres_sec / sum_of_preserved_area) * 100 if sum_of_preserved_area > 0 else 0
        
        perc_per_length = sec_perc / sec_length if sec_length > 0 else 0
        
        if perc_per_length > max_sec_perc:
            max_sec_perc = perc_per_length
        
        ax0_twin.hlines(perc_per_length, xmin=start, xmax=end, color='#B23A48', 
                        linestyle='-', linewidth=2.0, zorder=2)

        # 2. GREY DASHED LINE: Un-normalized mean of the bubbles (Left Axis)
        section_preservation = preservation_per_bend[start:end]
        if len(section_preservation) > 0:
            mean_preservation = np.mean(section_preservation)
            
            # Print metrics to console
            print(f"[Filtered] {name}:")
            print(f"    Mean Bend Preservation Rate: {mean_preservation:.2f}% (Plotted as dashed line)")
            print(f"    Share of Total River Preservation: {sec_perc:.2f}% (These will sum to 100%!)")
            print(f"    Preservation efficiency: {perc_per_length:.2f}% per km (Plotted as red solid line)")
            
            ax[0].hlines(mean_preservation, xmin=start, xmax=end, color='#444444', 
                         linestyle='--', linewidth=1.5, zorder=2, alpha=0.8)

        # 3. DRAW TOP BOUNDING ARROWS & TEXT
        arrow_y = 105
        text_y = 98

        ax[0].annotate('', xy=(start, arrow_y), xytext=(end, arrow_y), 
                       arrowprops=dict(arrowstyle='<->', ls='--', lw=1.0, color='black'), zorder=4)
        midpt = start + (end - start) / 2
        ax[0].text(midpt, text_y, name, ha='center', va='top', fontsize=7, fontweight='bold', zorder=5)

    # Vertical dividing lines
    ax[0].axvline(x=50, color='black', linestyle='--', linewidth=1.0, alpha=0.8, zorder=2)
    ax[0].axvline(x=70, color='black', linestyle='--', linewidth=1.0, alpha=0.8, zorder=2)

    ax0_twin.set_ylim(0, (max_sec_perc * 1.5) if max_sec_perc > 0 else 1)
    ax[0].set_xlim(0, num_bends + 1)

    # --- Subplot 2: Absolute Area per Event ---
    depo_area_km2 = depositional_area_per_age / 1e6
    pres_area_km2 = bar_poly_area_per_age / 1e6

    bars_1 = ax[1].bar(event_labels, depo_area_km2, color=custom_colors, edgecolor='black', linewidth=0.5, zorder=3)
    ax[1].bar(event_labels, pres_area_km2, color='white', alpha=0.4, edgecolor='black', linewidth=0.5, hatch='//////', zorder=4)
    ax[1].set_ylabel('Total depositional area (km$^2$)')
    ax[1].set_xlabel('Event')

    # Percentage annotations
    for bar, percentage in zip(bars_1, preservation_per_age):
        height = bar.get_height()
        ax[1].annotate(f'{percentage:.0f}%',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 2),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=7, zorder=5)

    max_depo_km2 = np.max(depo_area_km2) if len(depo_area_km2) > 0 else 1
    ax[1].set_ylim(0, max_depo_km2 * 1.2) 
    
    overlay_legend = [mpatches.Patch(facecolor='white', edgecolor='black', linewidth=0.5, hatch='//////', label='Preserved')]
    ax[1].legend(handles=overlay_legend, loc='upper right', fontsize=7, framealpha=1.0, edgecolor='black')

    # --- Subplot 3: Fraction of total preserved area ---
    bars_2 = ax[2].bar(event_labels, areas, color=custom_colors, edgecolor='black', linewidth=0.5, zorder=3)
    ax[2].set_ylabel('% of total preserved area')
    ax[2].set_xlabel('Event')

    def add_labels(bars, axis):
        for bar in bars:
            height = bar.get_height()
            axis.annotate(f'{height:.1f}%',
                          xy=(bar.get_x() + bar.get_width() / 2, height),
                          xytext=(0, 2),
                          textcoords="offset points",
                          ha='center', va='bottom', fontsize=7, zorder=5)

    add_labels(bars_2, ax[2])
    ax[2].set_ylim(0, 110)

    # --- Stacked Global Legend in Subplot 3 (Upper Left) ---
    bg_box = mpatches.Rectangle((0.02, 0.68), 0.62, 0.28, facecolor='white', edgecolor='black', linewidth=0.5, transform=ax[2].transAxes, zorder=5, alpha=0.95)
    ax[2].add_patch(bg_box)
    
    baseflow_keys = [0, 2, 4, 6, 8, 10, 12, 14]
    flood_keys = [1, 3, 5, 7, 9, 11, 13]
    txt_outline = [pe.withStroke(linewidth=0.8, foreground="black")]
    
    block_w = 0.05
    block_h = 0.10
    start_x = 0.20

    # Flood Row Legend
    ax[2].text(0.04, 0.86, 'Flood', transform=ax[2].transAxes, va='center', ha='left', fontsize=8, fontweight='bold', zorder=6)
    for i, k in enumerate(flood_keys):
        x_pos = start_x + i * block_w
        rect = mpatches.Rectangle((x_pos, 0.86 - block_h/2), block_w, block_h, facecolor=color_dict[k], edgecolor='black', linewidth=0.5, transform=ax[2].transAxes, zorder=6)
        ax[2].add_patch(rect)
        ax[2].text(x_pos + block_w/2, 0.86, f"F{i+1}", transform=ax[2].transAxes, color='white', ha='center', va='center', fontsize=7, fontweight='bold', path_effects=txt_outline, zorder=7)

    # Baseflow Row Legend
    ax[2].text(0.04, 0.75, 'Baseflow', transform=ax[2].transAxes, va='center', ha='left', fontsize=8, fontweight='bold', zorder=6)
    for i, k in enumerate(baseflow_keys):
        x_pos = start_x + i * block_w
        rect = mpatches.Rectangle((x_pos, 0.75 - block_h/2), block_w, block_h, facecolor=color_dict[k], edgecolor='black', linewidth=0.5, transform=ax[2].transAxes, zorder=6)
        ax[2].add_patch(rect)
        ax[2].text(x_pos + block_w/2, 0.75, f"B{i+1}", transform=ax[2].transAxes, color='white', ha='center', va='center', fontsize=7, fontweight='bold', path_effects=txt_outline, zorder=7)

    # Force all primary y-axis labels to align to the leftmost bounding box
    fig.align_ylabels(ax[:]) 
    
    # Apply tight layout with slightly more vertical padding
    fig.tight_layout(h_pad=1.2) 

    filepath_png = os.path.join(fig_dir, "Fig_12.png")
    filepath_pdf = os.path.join(fig_dir, "Fig_12.pdf")
    plt.savefig(filepath_png, dpi=500, bbox_inches='tight')
    plt.savefig(filepath_pdf, bbox_inches='tight')
    plt.show()